# Ablation Study: GSPO Training Techniques

Systematic evaluation of each technique's contribution to STEM reasoning quality.

**Research Questions:**
- **RQ1**: Does SFT initialization improve GSPO convergence vs Instruct-only? 
- **RQ2**: GSPO (sequence-level IS) vs GRPO (token-level) — which is better for Qwen3-4B?
- **RQ3**: Does curriculum learning (easy→hard) outperform random sampling?
- **RQ4**: Does ReDit reward dithering improve convergence speed?
- **RQ5**: Does Dr.GRPO length normalization prevent response length bias?
- **RQ6**: How do domain-specific accuracies change across ablation variants?

**Experiments:**

| ID | Variant | SFT | IS Level | Curriculum | ReDit | Dr.GRPO | Diff. Weights |
|---|---------|-----|----------|------------|-------|---------|---------------|
| E0 | Baseline (Instruct) | - | - | - | - | - | - |
| E1 | SFT only | Yes | - | - | - | - | - |
| E2 | GRPO (token-level) | No | token | No | No | No | No |
| E3 | GSPO (sequence-level) | No | sequence | No | No | No | No |
| E4 | GSPO + Curriculum | No | sequence | Yes | No | No | No |
| E5 | GSPO + Curriculum + ReDit | No | sequence | Yes | Yes | No | No |
| E6 | GSPO + All techniques | No | sequence | Yes | Yes | Yes | Yes |
| E7 | SFT → GSPO + All | Yes | sequence | Yes | Yes | Yes | Yes |

**Target hardware:** Google Colab A100 40GB / 80GB

**Evaluation:** 10% held-out test set, per-domain accuracy, reward curves

In [ ]:
# ============================================================
# Setup
# ============================================================
import os
os.environ["UNSLOTH_OFFLOAD_GRADIENTS"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

!pip install -q unsloth "trl>=0.27.0" peft transformers datasets
!pip install -q accelerate bitsandbytes sentencepiece protobuf
!pip install -q sympy chempy matplotlib seaborn pandas

from huggingface_hub import login
login()

In [ ]:
# ============================================================
# Mount Drive + add scripts to path
# ============================================================
import sys, os, json

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

DRIVE_ROOT = "/content/drive/MyDrive"
if DRIVE_ROOT not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)

# Verify scripts
for s in ["training/scripts/stem_rewards.py", "training/scripts/verify_answers.py",
          "training/scripts/sort_curriculum.py"]:
    path = os.path.join(DRIVE_ROOT, s)
    print(f"  {'OK' if os.path.exists(path) else 'MISSING'} {s}")

In [ ]:
# ============================================================
# Experiment Configurations
# ============================================================
from dataclasses import dataclass, field
from typing import Optional, List, Dict

# Shared constants
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SFT_HF_REPO = "Siesher/mits-qwen3-4b-sft"
OUTPUT_BASE = "/content/drive/MyDrive/MITS/experiments"
A100_VRAM_GB = 80

@dataclass
class ExperimentConfig:
    """Configuration for one ablation experiment."""
    name: str
    description: str
    # Model initialization
    use_sft_init: bool = False        # Start from SFT adapter
    # RL technique toggles
    run_rl: bool = True               # Whether to run RL training
    is_level: str = "sequence"         # "sequence" (GSPO) or "token" (GRPO)
    loss_type: str = "dr_grpo"        # "dr_grpo", "grpo"
    use_curriculum: bool = False       # Two-stage curriculum
    use_redit: bool = False            # ReDit reward dithering
    dithering_sigma: float = 0.05
    use_difficulty_weights: bool = False  # GRPO-LEAD reweighting
    # Clipping — GSPO uses tiny epsilon, GRPO uses standard
    epsilon: float = 3e-4
    epsilon_high: float = 4e-4
    # Training
    total_steps: int = 200            # Keep short for ablation
    lora_r: int = 16
    lora_alpha: int = 32
    learning_rate: float = 1e-6
    G: int = 16
    max_completion: int = 768
    max_prompt: int = 512
    gradient_accumulation: int = 2
    steps_per_generation: int = 8


EXPERIMENTS = {
    "E0_baseline": ExperimentConfig(
        name="E0_baseline",
        description="Instruct model without any training (baseline)",
        use_sft_init=False,
        run_rl=False,
    ),
    "E1_sft_only": ExperimentConfig(
        name="E1_sft_only",
        description="SFT adapter only, no RL training",
        use_sft_init=True,
        run_rl=False,
    ),
    "E2_grpo_token": ExperimentConfig(
        name="E2_grpo_token",
        description="Standard GRPO (token-level IS, standard clipping)",
        use_sft_init=False,
        is_level="token",
        loss_type="grpo",
        epsilon=0.2,       # Standard GRPO clipping
        epsilon_high=0.28, # Clip-Higher
    ),
    "E3_gspo_sequence": ExperimentConfig(
        name="E3_gspo_sequence",
        description="GSPO (sequence-level IS, tiny epsilon)",
        use_sft_init=False,
        is_level="sequence",
        loss_type="grpo",  # No Dr.GRPO yet
    ),
    "E4_gspo_curriculum": ExperimentConfig(
        name="E4_gspo_curriculum",
        description="GSPO + two-stage curriculum (easy/med → all)",
        use_sft_init=False,
        is_level="sequence",
        loss_type="grpo",
        use_curriculum=True,
    ),
    "E5_gspo_curriculum_redit": ExperimentConfig(
        name="E5_gspo_curriculum_redit",
        description="GSPO + Curriculum + ReDit reward dithering",
        use_sft_init=False,
        is_level="sequence",
        loss_type="grpo",
        use_curriculum=True,
        use_redit=True,
    ),
    "E6_gspo_all_techniques": ExperimentConfig(
        name="E6_gspo_all_techniques",
        description="GSPO + Curriculum + ReDit + Dr.GRPO + Difficulty weights",
        use_sft_init=False,
        is_level="sequence",
        loss_type="dr_grpo",
        use_curriculum=True,
        use_redit=True,
        use_difficulty_weights=True,
    ),
    "E7_sft_gspo_all": ExperimentConfig(
        name="E7_sft_gspo_all",
        description="SFT init → GSPO + all techniques (full pipeline)",
        use_sft_init=True,
        is_level="sequence",
        loss_type="dr_grpo",
        use_curriculum=True,
        use_redit=True,
        use_difficulty_weights=True,
    ),
}

print(f"Defined {len(EXPERIMENTS)} experiments:")
for eid, cfg in EXPERIMENTS.items():
    print(f"  {eid}: {cfg.description}")

In [ ]:
# ============================================================
# Load data + create held-out evaluation split
# ============================================================
import json
import random
import numpy as np
from collections import Counter
from datasets import load_dataset

RL_DATA_PATH = "/content/drive/MyDrive/training/data/rl_combined.jsonl"

def load_rl_problems():
    if os.path.exists(RL_DATA_PATH):
        problems = []
        with open(RL_DATA_PATH, "r", encoding="utf-8") as f:
            for line in f:
                problems.append(json.loads(line))
        return problems
    hf_ds = load_dataset("Siesher/mits-stem-training-data", "rl")
    return [dict(r) for r in hf_ds["train"]]

all_problems = load_rl_problems()
verifiable = [p for p in all_problems
              if p.get("type", "verifiable") == "verifiable"
              and p.get("answer_type", "numeric") != "conceptual"]
print(f"Total verifiable problems: {len(verifiable)}")

# --- Stratified held-out split (10%) ---
# Stratify by domain to ensure each domain is represented in test set
random.seed(42)
EVAL_FRACTION = 0.10

domain_buckets = {}
for p in verifiable:
    d = p.get("domain", "math")
    domain_buckets.setdefault(d, []).append(p)

train_problems = []
eval_problems = []

for domain, problems in domain_buckets.items():
    random.shuffle(problems)
    n_eval = max(1, int(len(problems) * EVAL_FRACTION))
    eval_problems.extend(problems[:n_eval])
    train_problems.extend(problems[n_eval:])

print(f"\nSplit (seed=42, {EVAL_FRACTION*100:.0f}% eval):")
print(f"  Train: {len(train_problems)}")
print(f"  Eval:  {len(eval_problems)}")

# Per-domain eval counts
eval_domain_counts = Counter(p.get("domain", "unknown") for p in eval_problems)
print(f"\nEval set by domain:")
for domain, count in sorted(eval_domain_counts.items()):
    total = len(domain_buckets.get(domain, []))
    print(f"  {domain}: {count}/{total} ({count/total*100:.1f}%)")

# Classify difficulty for curriculum experiments
try:
    from training.scripts.sort_curriculum import classify_difficulty
except ImportError:
    def classify_difficulty(ex):
        text = ex.get("answer", "") + " " + ex.get("prompt", "")
        wc = len(text.split())
        if wc < 50: return "easy"
        elif wc > 150: return "hard"
        return "medium"

for p in train_problems:
    p["difficulty"] = classify_difficulty(p)
for p in eval_problems:
    p["difficulty"] = classify_difficulty(p)

diff_dist = Counter(p["difficulty"] for p in train_problems)
print(f"\nTrain difficulty: {dict(diff_dist)}")

In [ ]:
# ============================================================
# Model loading utility
# ============================================================
import torch
from unsloth import FastLanguageModel

TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

def load_model(config: 'ExperimentConfig'):
    """Load model based on experiment config.
    Returns (model, tokenizer).
    """
    if config.use_sft_init:
        # Load base + SFT adapter via Unsloth auto-detection
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=SFT_HF_REPO,
            max_seq_length=config.max_completion + config.max_prompt,
            load_in_4bit=True,
            dtype=torch.bfloat16,
        )
        if hasattr(model, 'peft_config'):
            _cfg = list(model.peft_config.values())[0]
            print(f"  SFT adapter: r={_cfg.r}, alpha={_cfg.lora_alpha}")
        if config.run_rl:
            model.gradient_checkpointing_enable()
    else:
        # Fresh model + LoRA
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=BASE_MODEL,
            max_seq_length=config.max_completion + config.max_prompt,
            load_in_4bit=True,
            dtype=torch.bfloat16,
        )
        if config.run_rl:
            model = FastLanguageModel.get_peft_model(
                model,
                r=config.lora_r,
                lora_alpha=config.lora_alpha,
                lora_dropout=0.0,
                target_modules=TARGET_MODULES,
                use_gradient_checkpointing="unsloth",
                random_state=42,
            )
    
    if hasattr(model, 'hf_device_map'):
        model.hf_device_map = {'': 0}
    
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Params: {trainable:,} trainable / {total:,} total")
    return model, tokenizer

print("Model loading utility ready.")

In [ ]:
# ============================================================
# Evaluation function
# ============================================================
import re
import random
from collections import defaultdict

# Import verification module
_verify_imported = False
try:
    from training.scripts.verify_answers import verify, extract_answer
    _verify_imported = True
    print("Imported verify, extract_answer from training.scripts.verify_answers")
except ImportError:
    print("WARNING: verify_answers not available, using fallback")

    def extract_answer(text):
        """Fallback answer extraction."""
        if "</think>" in text:
            text = text.split("</think>")[-1].strip()
        boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
        if boxed:
            return boxed[-1].strip()
        gsm = re.search(r'####\s*(.+?)$', text.strip(), re.MULTILINE)
        if gsm:
            return gsm.group(1).strip().replace(",", "")
        answer_ru = re.search(r'(?:Ответ|ответ)\s*[:=]\s*(.+?)(?:\.|$)', text)
        if answer_ru:
            return answer_ru.group(1).strip()
        lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
        return lines[-1] if lines else ""

    def _fallback_verify(answer, truth, domain, **kwargs):
        """Simple string/numeric comparison fallback."""
        import math
        a, t = answer.strip().lower(), truth.strip().lower()
        if a == t:
            return True
        try:
            return math.isclose(float(a.replace(",", ".")), float(t.replace(",", ".")), rel_tol=0.02)
        except (ValueError, TypeError):
            return False


SYSTEM_PROMPT_CALC = "Ты — репетитор по STEM. Реши задачу пошагово и запиши финальный ответ в \\boxed{}."
SYSTEM_PROMPT_MC = "Проанализируй задачу и выбери правильный ответ (A, B, C или D)."


def eval_model(model, tokenizer, eval_problems, max_samples=200, batch_size=4):
    """Evaluate model on held-out problems using greedy decoding.

    Generates one completion per problem, extracts the answer, and verifies
    correctness using domain-specific verification (verify_answers.py).

    Args:
        model: Loaded Unsloth/HF model (will be switched to inference mode)
        tokenizer: Associated tokenizer
        eval_problems: List of problem dicts with prompt, answer, domain fields
        max_samples: Max problems to evaluate (random subsample if exceeded)
        batch_size: Number of prompts per generation batch (left-padded)

    Returns:
        dict with overall_accuracy, per_domain stats, total_evaluated,
        avg_completion_length, and sample completions for inspection
    """
    # Subsample if needed (stratified by domain for fairness)
    if len(eval_problems) > max_samples:
        problems = random.sample(eval_problems, max_samples)
    else:
        problems = list(eval_problems)

    # Switch to fast inference mode (2x speedup via Unsloth)
    FastLanguageModel.for_inference(model)

    # Configure tokenizer for left-padded batching
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Format all prompts
    formatted = []
    for p in problems:
        sys_prompt = SYSTEM_PROMPT_MC if p.get("answer_type") == "mc_letter" else SYSTEM_PROMPT_CALC
        messages = [
            {"role": "system", "content": sys_prompt},
            {"role": "user", "content": p["prompt"]},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        formatted.append(text)

    # Generate completions in batches
    all_completions = []
    for batch_start in range(0, len(formatted), batch_size):
        batch_prompts = formatted[batch_start : batch_start + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        ).to(model.device)

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=768,
                do_sample=False,       # Greedy — deterministic for reproducibility
                temperature=None,      # Required when do_sample=False
                top_p=None,
                use_cache=True,
            )

        # Decode only the NEW tokens (strip the prompt)
        for i, prompt_text in enumerate(batch_prompts):
            prompt_len = inputs["input_ids"][i].shape[0]
            completion_ids = output_ids[i][prompt_len:]
            completion = tokenizer.decode(completion_ids, skip_special_tokens=True)
            all_completions.append(completion)

        # Progress
        done = min(batch_start + batch_size, len(formatted))
        if done % 20 == 0 or done == len(formatted):
            print(f"  Generated {done}/{len(formatted)}")

    # Verify answers
    per_domain = defaultdict(lambda: {"correct": 0, "total": 0})
    sample_completions = []  # Store first few for manual inspection
    completion_lengths = []

    for problem, completion in zip(problems, all_completions):
        domain = problem.get("domain", "math")
        truth = problem.get("ground_truth", problem.get("answer", ""))
        answer_type = problem.get("answer_type", "numeric")
        q_type = "mc" if answer_type == "mc_letter" else problem.get("type", "calc")

        extracted = extract_answer(completion)
        completion_lengths.append(len(completion.split()))

        if _verify_imported:
            result = verify(
                answer=extracted,
                truth=truth,
                domain=domain,
                question_type=q_type,
                test_cases=problem.get("test_cases"),
            )
            correct = result.correct
        else:
            correct = _fallback_verify(extracted, truth, domain)

        per_domain[domain]["total"] += 1
        if correct:
            per_domain[domain]["correct"] += 1

        # Save first 10 completions for manual review
        if len(sample_completions) < 10:
            sample_completions.append({
                "domain": domain,
                "prompt": problem["prompt"][:100] + "...",
                "truth": truth,
                "extracted": extracted,
                "correct": correct,
                "completion_length": len(completion.split()),
            })

    # Compute accuracies
    total_correct = sum(d["correct"] for d in per_domain.values())
    total_eval = sum(d["total"] for d in per_domain.values())

    per_domain_stats = {}
    for domain, stats in per_domain.items():
        per_domain_stats[domain] = {
            "correct": stats["correct"],
            "total": stats["total"],
            "accuracy": stats["correct"] / stats["total"] if stats["total"] > 0 else 0.0,
        }

    # Restore tokenizer and model state
    tokenizer.padding_side = original_padding_side
    # Restore training mode for potential continued RL
    try:
        FastLanguageModel.for_training(model)
    except Exception:
        pass  # OK if model is eval-only (E0, E1)

    result = {
        "overall_accuracy": total_correct / total_eval if total_eval > 0 else 0.0,
        "per_domain": per_domain_stats,
        "total_evaluated": total_eval,
        "avg_completion_length": sum(completion_lengths) / len(completion_lengths) if completion_lengths else 0,
        "sample_completions": sample_completions,
    }

    # Print summary
    print(f"  Accuracy: {result['overall_accuracy']:.1%} ({total_correct}/{total_eval})")
    print(f"  Avg completion: {result['avg_completion_length']:.0f} words")
    for domain in sorted(per_domain_stats.keys()):
        s = per_domain_stats[domain]
        print(f"    {domain}: {s['accuracy']:.1%} ({s['correct']}/{s['total']})")

    return result


print("Evaluation function ready.")

In [ ]:
# ============================================================
# Reward functions factory (per-experiment configuration)
# ============================================================
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset

try:
    from training.scripts.stem_rewards import make_gdpo_reward_fns
    _stem_imported = True
except ImportError:
    _stem_imported = False


def build_reward_funcs(config, problems, tokenizer):
    """Build reward functions based on experiment config."""
    sigma = config.dithering_sigma if config.use_redit else 0.0

    if _stem_imported:
        fns = make_gdpo_reward_fns(
            problems, tokenizer, SYSTEM_PROMPT_CALC,
            dithering_sigma=sigma,
        )
        correctness_fn = fns[0]
        format_fn = fns[1]
    else:
        raise RuntimeError("stem_rewards.py required for training")

    # Wrap with difficulty weighting if enabled
    if config.use_difficulty_weights:
        DIFF_WEIGHTS = {"easy": 0.5, "medium": 1.0, "hard": 2.0}
        # Build prompt-to-problem lookup for difficulty
        p2p = {}
        for p in problems:
            sys_p = SYSTEM_PROMPT_MC if p.get("answer_type") == "mc_letter" else SYSTEM_PROMPT_CALC
            msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": p["prompt"]}]
            fmt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            p2p[fmt.strip()] = p

        _base_fn = correctness_fn
        def weighted_fn(completions, prompts=None, **kw):
            rewards = _base_fn(completions, prompts=prompts, **kw)
            if prompts is None:
                return rewards
            weighted = []
            for r, pt in zip(rewards, prompts):
                prob = p2p.get(pt.strip())
                w = DIFF_WEIGHTS.get(prob["difficulty"], 1.0) if prob else 1.0
                weighted.append(r * w)
            return weighted
        correctness_fn = weighted_fn

    return [correctness_fn, format_fn]


def format_problems_as_dataset(problem_list, tokenizer):
    """Format problems into HF Dataset for GRPOTrainer."""
    formatted = []
    for p in problem_list:
        sys_p = SYSTEM_PROMPT_MC if p.get("answer_type") == "mc_letter" else SYSTEM_PROMPT_CALC
        msgs = [{"role": "system", "content": sys_p}, {"role": "user", "content": p["prompt"]}]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        formatted.append({"prompt": prompt})
    return Dataset.from_list(formatted)


print("Reward functions factory ready.")

In [ ]:
# ============================================================
# Experiment Runner
# ============================================================
import inspect
import time
import gc

RESULTS = {}  # Persists across cell re-runs


def make_grpo_config(config, output_dir, max_steps, warmup_ratio=0.05):
    """Create GRPOConfig from experiment config."""
    all_kwargs = dict(
        output_dir=output_dir,
        max_steps=max_steps,
        per_device_train_batch_size=config.G,
        gradient_accumulation_steps=config.gradient_accumulation,
        learning_rate=config.learning_rate,
        lr_scheduler_type="cosine",
        warmup_ratio=warmup_ratio,
        num_generations=config.G,
        max_completion_length=config.max_completion,
        max_prompt_length=config.max_prompt,
        loss_type=config.loss_type,
        beta=0.0,
        epsilon=config.epsilon,
        epsilon_high=config.epsilon_high,
        importance_sampling_level=config.is_level,
        steps_per_generation=config.steps_per_generation,
        mask_truncated_completions=True,
        reward_weights=[0.8, 0.2],
        bf16=True,
        logging_steps=5,
        save_steps=50,
        save_total_limit=2,
        optim="adamw_torch_fused",
        max_grad_norm=1.0,
        temperature=0.9,
        seed=42,
        report_to="none",
    )
    # Filter to params accepted by GRPOConfig.__init__
    sig = inspect.signature(GRPOConfig.__init__)
    valid = set(sig.parameters.keys())
    init_kw = {k: v for k, v in all_kwargs.items() if k in valid}
    post_kw = {k: v for k, v in all_kwargs.items() if k not in valid}
    cfg = GRPOConfig(**init_kw)
    for k, v in post_kw.items():
        setattr(cfg, k, v)
    return cfg


def run_experiment(exp_id: str, skip_if_exists=True):
    """Run a single ablation experiment."""
    if skip_if_exists and exp_id in RESULTS:
        print(f"\n{'='*60}")
        print(f"SKIP {exp_id}: already in RESULTS")
        print(f"{'='*60}")
        return RESULTS[exp_id]

    config = EXPERIMENTS[exp_id]
    output_dir = os.path.join(OUTPUT_BASE, exp_id)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"EXPERIMENT: {exp_id}")
    print(f"  {config.description}")
    print(f"{'='*60}")

    t0 = time.time()

    # 1. Load model
    print("\n[1/3] Loading model...")
    model, tokenizer = load_model(config)

    # 2. Train (if RL is enabled)
    training_loss = None
    if config.run_rl:
        print(f"\n[2/3] Training ({config.total_steps} steps, {config.is_level} IS, {config.loss_type})...")

        if config.use_curriculum:
            # Stage 1: easy + medium
            s1_problems = [p for p in train_problems if p["difficulty"] in ("easy", "medium")]
            s1_steps = config.total_steps // 3  # 1/3 warmup
            s1_ds = format_problems_as_dataset(s1_problems, tokenizer)
            s1_cfg = make_grpo_config(config, os.path.join(output_dir, "s1"), s1_steps, warmup_ratio=0.1)
            reward_fns = build_reward_funcs(config, s1_problems, tokenizer)
            trainer = GRPOTrainer(model=model, args=s1_cfg, train_dataset=s1_ds,
                                 reward_funcs=reward_fns, processing_class=tokenizer)
            print(f"  Stage 1: {len(s1_problems)} easy+medium, {s1_steps} steps")
            result_s1 = trainer.train()
            del trainer; torch.cuda.empty_cache()

            # Stage 2: all
            s2_steps = config.total_steps - s1_steps
            s2_ds = format_problems_as_dataset(train_problems, tokenizer)
            s2_cfg = make_grpo_config(config, os.path.join(output_dir, "s2"), s2_steps, warmup_ratio=0.03)
            reward_fns = build_reward_funcs(config, train_problems, tokenizer)
            trainer = GRPOTrainer(model=model, args=s2_cfg, train_dataset=s2_ds,
                                 reward_funcs=reward_fns, processing_class=tokenizer)
            print(f"  Stage 2: {len(train_problems)} all, {s2_steps} steps")
            result_s2 = trainer.train()
            training_loss = result_s2.training_loss
            del trainer; torch.cuda.empty_cache()
        else:
            # Single-stage training
            ds = format_problems_as_dataset(train_problems, tokenizer)
            grpo_cfg = make_grpo_config(config, output_dir, config.total_steps)
            reward_fns = build_reward_funcs(config, train_problems, tokenizer)
            trainer = GRPOTrainer(model=model, args=grpo_cfg, train_dataset=ds,
                                 reward_funcs=reward_fns, processing_class=tokenizer)
            print(f"  Single stage: {len(train_problems)} problems, {config.total_steps} steps")
            result = trainer.train()
            training_loss = result.training_loss
            del trainer; torch.cuda.empty_cache()
    else:
        print("\n[2/3] No RL training (eval-only experiment)")

    # 3. Evaluate on held-out set
    print(f"\n[3/3] Evaluating on {len(eval_problems)} held-out problems...")
    eval_result = eval_model(model, tokenizer, eval_problems)

    elapsed = time.time() - t0

    # Save results
    result = {
        "config": config.__dict__,
        "training_loss": training_loss,
        "eval": eval_result,
        "elapsed_seconds": elapsed,
    }
    RESULTS[exp_id] = result

    # Save to disk
    results_path = os.path.join(output_dir, "results.json")
    with open(results_path, "w") as f:
        json.dump({k: v for k, v in result.items() if k != "eval" or "completions" not in v},
                  f, indent=2, default=str)

    # Save adapter
    if config.run_rl:
        model.save_pretrained(os.path.join(output_dir, "adapter"))
        tokenizer.save_pretrained(os.path.join(output_dir, "adapter"))

    # Cleanup
    del model
    gc.collect()
    torch.cuda.empty_cache()

    print(f"\n{exp_id} complete in {elapsed/60:.1f} min")
    print(f"  Overall accuracy: {eval_result['overall_accuracy']:.1%}")
    if training_loss:
        print(f"  Training loss: {training_loss:.4f}")
    return result


print("Experiment runner ready.")

In [ ]:
# ============================================================
# Run all experiments
# ============================================================
# Run experiments in order. Each run loads a fresh model,
# trains (if applicable), evaluates, and saves results.
# Skip already-completed experiments with skip_if_exists=True.

EXPERIMENT_ORDER = [
    "E0_baseline",
    "E1_sft_only",
    "E2_grpo_token",
    "E3_gspo_sequence",
    "E4_gspo_curriculum",
    "E5_gspo_curriculum_redit",
    "E6_gspo_all_techniques",
    "E7_sft_gspo_all",
]

for exp_id in EXPERIMENT_ORDER:
    try:
        run_experiment(exp_id)
    except NotImplementedError as e:
        print(f"\n  STOPPED: {e}")
        print("  Implement eval_model() first, then re-run this cell.")
        break
    except Exception as e:
        print(f"\n  ERROR in {exp_id}: {e}")
        import traceback; traceback.print_exc()
        continue

In [ ]:
# ============================================================
# Results Visualization
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

if not RESULTS:
    print("No results yet — run experiments first.")
else:
    # --- Table: Overall accuracy ---
    rows = []
    for eid, res in RESULTS.items():
        row = {
            "Experiment": eid,
            "Accuracy": res["eval"]["overall_accuracy"],
            "Loss": res.get("training_loss"),
            "Time (min)": res["elapsed_seconds"] / 60,
        }
        # Per-domain
        for domain, stats in res["eval"].get("per_domain", {}).items():
            row[f"acc_{domain}"] = stats["accuracy"]
        rows.append(row)

    df = pd.DataFrame(rows).set_index("Experiment")
    print("\n=== Ablation Results ===")
    print(df.to_string(float_format="{:.3f}".format))

    # --- Bar chart: Overall accuracy ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Overall accuracy
    ax = axes[0]
    experiments = list(RESULTS.keys())
    accuracies = [RESULTS[e]["eval"]["overall_accuracy"] for e in experiments]
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(experiments)))
    ax.bar(range(len(experiments)), accuracies, color=colors)
    ax.set_xticks(range(len(experiments)))
    ax.set_xticklabels([e.split("_", 1)[0] for e in experiments], rotation=45)
    ax.set_ylabel("Accuracy")
    ax.set_title("Overall Accuracy by Experiment")
    ax.set_ylim(0, 1)
    for i, v in enumerate(accuracies):
        ax.text(i, v + 0.01, f"{v:.1%}", ha="center", fontsize=9)

    # Per-domain accuracy heatmap
    ax = axes[1]
    domains = ["math", "physics", "chemistry", "biology", "cs"]
    domain_data = []
    for e in experiments:
        per_domain = RESULTS[e]["eval"].get("per_domain", {})
        domain_data.append([per_domain.get(d, {}).get("accuracy", 0) for d in domains])
    
    im = ax.imshow(domain_data, cmap="YlOrRd", aspect="auto", vmin=0, vmax=1)
    ax.set_xticks(range(len(domains)))
    ax.set_xticklabels(domains)
    ax.set_yticks(range(len(experiments)))
    ax.set_yticklabels([e.split("_", 1)[0] for e in experiments])
    ax.set_title("Per-Domain Accuracy")
    plt.colorbar(im, ax=ax)
    # Annotate cells
    for i in range(len(experiments)):
        for j in range(len(domains)):
            ax.text(j, i, f"{domain_data[i][j]:.0%}", ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_BASE, "ablation_results.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"\nPlot saved to {OUTPUT_BASE}/ablation_results.png")

In [ ]:
# ============================================================
# Incremental accuracy analysis (delta from baseline)
# ============================================================
if len(RESULTS) >= 2 and "E0_baseline" in RESULTS:
    baseline_acc = RESULTS["E0_baseline"]["eval"]["overall_accuracy"]
    baseline_domain = RESULTS["E0_baseline"]["eval"].get("per_domain", {})

    print("\n=== Incremental Contribution (delta from E0 baseline) ===")
    print(f"{'Experiment':<30} {'Accuracy':>10} {'Delta':>10} {'Technique added':>30}")
    print("-" * 85)

    technique_desc = {
        "E0_baseline": "(baseline)",
        "E1_sft_only": "+SFT init",
        "E2_grpo_token": "+GRPO (token IS)",
        "E3_gspo_sequence": "+GSPO (sequence IS)",
        "E4_gspo_curriculum": "+Curriculum",
        "E5_gspo_curriculum_redit": "+ReDit dithering",
        "E6_gspo_all_techniques": "+Dr.GRPO + diff weights",
        "E7_sft_gspo_all": "+SFT init (full pipeline)",
    }

    for eid in EXPERIMENT_ORDER:
        if eid not in RESULTS:
            continue
        acc = RESULTS[eid]["eval"]["overall_accuracy"]
        delta = acc - baseline_acc
        desc = technique_desc.get(eid, "")
        sign = "+" if delta > 0 else ""
        print(f"{eid:<30} {acc:>9.1%} {sign}{delta:>9.1%} {desc:>30}")

    # Per-domain deltas
    print("\n=== Per-Domain Deltas (vs E0 baseline) ===")
    domains = ["math", "physics", "chemistry", "biology", "cs"]
    header = f"{'Experiment':<30}" + "".join(f"{d:>12}" for d in domains)
    print(header)
    print("-" * (30 + 12 * len(domains)))
    for eid in EXPERIMENT_ORDER:
        if eid not in RESULTS:
            continue
        per_d = RESULTS[eid]["eval"].get("per_domain", {})
        deltas = []
        for d in domains:
            acc = per_d.get(d, {}).get("accuracy", 0)
            base = baseline_domain.get(d, {}).get("accuracy", 0)
            delta = acc - base
            sign = "+" if delta > 0 else ""
            deltas.append(f"{sign}{delta:.1%}")
        print(f"{eid:<30}" + "".join(f"{d:>12}" for d in deltas))
else:
    print("Need at least E0_baseline + one other experiment for delta analysis.")